In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
import torch.nn as nn
# set once — new tensors land on MPS without device= every call
if torch.backends.mps.is_available():
    torch.set_default_device("cpu")
torch.set_printoptions(sci_mode=False)


In [2]:
words = lambda: open("names.txt", 'r').read().splitlines()
stoi = {s:i+1 for i, s in enumerate(sorted(set(''.join(words()))))} | {'.':0,}
itos = {i:s for s, i in stoi.items()}
vocab_size = len(stoi)
block_size = 4


In [3]:
import random

def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0]*block_size
        for w in word+'.':
            idx = stoi[w]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx]
    return torch.tensor(X), torch.tensor(Y)

all_words = words()
random.seed(42)
random.shuffle(all_words)

n1 = int(0.8*len(all_words))
n2 = int(0.9*len(all_words))

Xtr, Ytr = build_dataset(all_words[:n1])      # 80% train
Xdev, Ydev = build_dataset(all_words[n1:n2])  # 10% dev
Xte, Yte = build_dataset(all_words[n2:])      # 10% test

print("train:", Xtr.shape[0], " dev:", Xdev.shape[0], " test:", Xte.shape[0])


train: 182625  dev: 22655  test: 22866


In [4]:
embed_dim = 10
hidden_size = 100
n_embed_total = block_size * embed_dim  # flattened size of block_size embedded characters

model = nn.Sequential(
    nn.Embedding(vocab_size, embed_dim), nn.Flatten(),
    nn.Linear(n_embed_total, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, hidden_size, bias=False), nn.BatchNorm1d(hidden_size), nn.Tanh(),
    nn.Linear(hidden_size, vocab_size, bias=False), nn.BatchNorm1d(vocab_size)
)


In [5]:
parameters = list(model.parameters())
print(sum(p.nelement() for p in parameters))


58224


In [6]:
model.train()
g = torch.Generator().manual_seed(2147483647)

for _ in tqdm(range(50_000)):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)

    logits = model(Xtr[ix])
    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.01
    for p in parameters:
        p.data += -lr*p.grad

print(loss.item())


  0%|          | 0/50000 [00:00<?, ?it/s]

  0%|          | 28/50000 [00:00<03:01, 275.20it/s]

  0%|          | 59/50000 [00:00<02:50, 292.83it/s]

  0%|          | 91/50000 [00:00<02:44, 302.60it/s]

  0%|          | 122/50000 [00:00<02:43, 304.36it/s]

...

100%|█████████▉| 49914/50000 [02:44<00:00, 273.47it/s]

100%|█████████▉| 49945/50000 [02:44<00:00, 281.12it/s]

100%|█████████▉| 49976/50000 [02:44<00:00, 287.11it/s]

100%|██████████| 50000/50000 [02:44<00:00, 303.45it/s]

2.0382838249206543


In [7]:
@torch.no_grad()
def split_loss(X, Y):
    model.eval()
    logits = model(X)
    return F.cross_entropy(logits, Y).item()

print("train loss:", split_loss(Xtr, Ytr))
print("dev loss:  ", split_loss(Xdev, Ydev))


train loss: 2.062187433242798
dev loss:   2.1071105003356934


In [8]:
model.eval()
with torch.no_grad():
    for _ in range(20):
        context = [0]*block_size
        out = []
        while True:
            logits = model(torch.tensor([context]))
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            if not ix: break
            out += [itos[ix]]
        print("".join(out))


arysson
blaun
blasonnis
kairaja
chrise
shlyn
shaenniray
kert
dardytia
martybe
kraylena
samara
eyca
ivah
kaelenawa
jarren
yalam
beatance
grah
ke
